# Hafta 3 — Gelişmiş RAG: Yeniden Sıralama ve Sorgu Dönüşümü

> [`src/reranking/`](../src/reranking/) **için kaynak not defteri.**

## Öğrenme hedefleri

1. Bi-Encoder ile Cross-Encoder arasındaki farkı biçimsel olarak tanımlamak ve aralarındaki gecikme/kalite Pareto sınırını türetmek.
2. Üç sorgu dönüşüm stratejisini uygulamak ve üzerlerinde akıl yürütmek: yeniden yazma, ayrıştırma ve Varsayımsal Belge Gömmeleri (HyDE).
3. Cross-encoder'ları eğitmek için kullanılan noktasal BCE ve ikili marj kayıplarını türetmek ve her ikisini NumPy ile uygulamak.
4. Hafta 2 RAG hattına bir Cross-Encoder yeniden sıralama aşaması eklemek ve nDCG@k üzerindeki artışı ölçmek.

> **Not:** Örnek derlem ve sorgular, kullanılan gömme ve cross-encoder modelleri İngilizce olduğu için İngilizce bırakılmıştır.

## 1. Bi-Encoder darboğazı

Bir Bi-Encoder, erişim skorunu iki bağımsız gömmenin nokta çarpımı olarak tanımlar:

$$
s_{\text{bi}}(q, d) = \phi(q)^\top \phi(d), \qquad \phi:\Sigma^\* \to \mathbb{R}^d.
$$

$\phi(d)$ yalnızca $d$'ye bağlı olduğundan tüm belge gömmeleri önceden hesaplanıp bir ANN indeksinde saklanabilir. Sorgu başına maliyet $O(d \cdot \log N)$'dir — hızlıdır, ancak katı bir doğruluk tavanı vardır: sorgu ve belgenin kodlama sırasında **etkileşime girme fırsatı yoktur**.

Bir Cross-Encoder çifti birleştirir ve skaler bir değer tahmin eder:

$$
s_{\text{cross}}(q, d) = f_\theta\bigl([\text{CLS}]\,q\,[\text{SEP}]\,d\bigr) \in \mathbb{R}.
$$

Sorgu ve belge token'ları arasındaki ortak dikkat, Bi-Encoder'ın yakalayamadığı ince taneli etkileşimleri (olumsuzluk, gönderim/anafora, birebir eşleşen anahtar sözcükler) yakalar. Bedeli: $\phi(d)$ önceden hesaplanamaz, dolayısıyla her sorgu bir Transformer üzerinden $N$ ileri geçiş gerektirir.

**Bu yüzden her ciddi erişim sistemi iki aşamalıdır**:

```
Bi-Encoder    →  ilk 50 aday              (ucuz, paralel, geri çağırma odaklı)
Cross-Encoder →  yeniden sıralı ilk 5     (pahalı, sıralı, kesinlik odaklı)
```

In [ ]:
# Gecikme ödünleşiminin büyüklük mertebesi tahmini.
N_docs        = 1_000_000
d_emb         = 384
bi_per_doc_us = 0.3          # ANN araması graf gezintisinin baskınlığında
ce_per_pair_ms = 8.0          # CPU üzerinde ms-marco-MiniLM-L-6

bi_total_ms = N_docs * bi_per_doc_us / 1000 / 50   # HNSW büyük ölçüde budar
ce_50_ms    = 50 * ce_per_pair_ms
ce_all_ms   = N_docs * ce_per_pair_ms

print(f"1M belge üzerinde Bi-Encoder (HNSW):          ~{bi_total_ms:6.0f} ms")
print(f"İlk 50 aday üzerinde Cross-Encoder:          ~{ce_50_ms:6.0f} ms")
print(f"1M belge üzerinde Cross-Encoder (imkânsız):  ~{ce_all_ms:6.0f} ms ({ce_all_ms/1000/60:.0f} dakika)")


## 2. Sorgu yeniden yazma ve ayrıştırma

Bi-Encoder sorguları bir *soru manifolduna*, belgeleri ise bir *yanıt manifolduna* gömer. İki manifold arasındaki boşluk, erişim kaçırmalarının büyük bir kısmından sorumludur. LLM güdümlü üç hafifletme yöntemi:

### 2a. Yeniden yazma

Kullanıcı sorgusunu derlemde geçme olasılığı daha yüksek terminolojiyle yeniden ifade edin. Kısaltmaları açın; gündelik ifadeleri teknik terimlerle değiştirin.

> *"RAG daha isabetli olabilir mi"* → *"erişimle zenginleştirilmiş üretim sistemlerinin kesinliğini ve geri çağırmasını iyileştirme teknikleri"*

### 2b. Ayrıştırma

Çok adımlı bir soruyu, birleşimi için erişimi daha kolay olan bağımsız tek adımlı alt sorulara bölün.

> *"HyDE'yi geliştirenler BERT üzerinde de çalıştı mı?"*
> → ["HyDE'nin yazarları kimlerdir?", "BERT'in yazarları kimlerdir?"]

### 2c. Varsayımsal Belge Gömmeleri (HyDE)

LLM'den sorgu için *makul bir yanıt belgesi halüsinasyonu üretmesini* isteyin, ardından sorgu yerine yanıtı gömün. Olgusal hatalar içerse bile uydurulmuş yanıtın gömmesi, gerçek yanıtlara orijinal sorgunun gömmesinden daha yakındır.

Biçimsel olarak:

$$
s_{\text{HyDE}}(q, d) = \phi\bigl(\text{LLM}(q)\bigr)^\top \phi(d).
$$

$T>0$ sıcaklığında örneklenen $K$ hipotezle:

$$
\hat\phi_q = \frac{1}{K}\sum_{k=1}^K \phi\bigl(\text{LLM}_k(q)\bigr).
$$

Konu dışı halüsinasyonlar geri çağırmayı yok edemesin diye ortalamaya ayrıca $\phi(q)$'yu da dâhil ederiz.

In [ ]:
# Deterministik sahte bir LLM ile gösterim (not defterinde gerçek API çağrısı yok).
# Üretimde bu `AnthropicClient` veya `OpenAIClient` olur.
from dataclasses import dataclass
from src.utils.llm_client import Message

class FakeLLM:
    def __init__(self, scripted):
        self.scripted = list(scripted)
    def complete(self, messages, **kw):
        return self.scripted.pop(0)

from src.reranking.query_rewriter import QueryRewriter

rewriter = QueryRewriter(FakeLLM([
    '{"rewritten": "techniques for improving precision of retrieval-augmented generation"}',
    '{"sub_questions": ["Who invented HyDE?", "Who invented BERT?"]}',
]))

print("yeniden yazma:", rewriter.rewrite("can RAG be more accurate"))
print("ayrıştırma:", rewriter.decompose("Did the inventors of HyDE also work on BERT?"))


In [ ]:
import numpy as np
from src.reranking.hyde import HyDE
from src.rag_engine.embeddings import SentenceEncoder

hyde = HyDE(
    llm=FakeLLM(["Sentence-Bert dense retrieval can be improved with hierarchical chunking and cross-encoder re-ranking, which jointly score query-document pairs."]),
    encoder=SentenceEncoder("BAAI/bge-small-en-v1.5"),
    n_hypotheses=1,
)
qvec = hyde.embed_query("how can dense retrieval be improved")
print("HyDE sorgu vektörü boyutu:", qvec.shape, "norm:", float(np.linalg.norm(qvec)))


## 3. Cross-Encoder eğitim hedefleri

Yeniden sıralayıcıları eğitmek için yaygın olarak iki kayıp kullanılır. Her ikisi de [`src/reranking/cross_encoder.py`](../src/reranking/cross_encoder.py) dosyasında uygulanmıştır.

### 3a. Noktasal BCE

İlgililik etiketleri $y_i \in \{0, 1\}$ ve ham skorlar $s_i$ verildiğinde:

$$
\mathcal{L}_{\text{BCE}} = -\frac{1}{N}\sum_{i=1}^N \Bigl(y_i \log \sigma(s_i) + (1 - y_i)\log(1 - \sigma(s_i))\Bigr).
$$

Bu, yeniden sıralamayı ikili sınıflandırma olarak ele alır: *$i$ belgesi sorguyla ilgili mi?* Eğitmesi kolaydır, ancak pozitiflerin kendi aralarındaki *sıralamasına* kayıtsızdır.

### 3b. İkili marj (menteşe / hinge)

Bir pozitif $(q, d^+)$ ve bir negatif $(q, d^-)$ verildiğinde bir marj zorlanır:

$$
\mathcal{L}_{\text{margin}} = \frac{1}{N}\sum_{i=1}^N \max\bigl(0,\, m - (s_i^+ - s_i^-)\bigr).
$$

Bu, erişimin önemsediği şey olan *sıralamayı* doğrudan optimize eder, ancak $(q, d^+, d^-)$ üçlülerinin oluşturulmasını gerektirir — bu genellikle Bi-Encoder'ın ilk-k sonuçları üzerinde zor negatif madenciliği (hard-negative mining) ile yapılır.

In [ ]:
import numpy as np
from src.reranking.cross_encoder import pointwise_bce_loss, pairwise_margin_loss

# Noktasal: kusursuz tahminler → ~0 kayıp.
scores = np.array([ 5.0,  5.0, -5.0, -5.0])
labels = np.array([ 1.0,  1.0,  0.0,  0.0])
print(f"BCE (kusursuz):  {pointwise_bce_loss(scores, labels):.6f}")

# Noktasal: ters tahminler → büyük kayıp.
print(f"BCE (ters):      {pointwise_bce_loss(-scores, labels):.4f}")

# İkili: pozitifler negatiflerin açıkça üstünde → 0 kayıp.
pos = np.array([3.0, 3.0]); neg = np.array([0.0, 0.0])
print(f"Marj (ayrık):    {pairwise_margin_loss(pos, neg, margin=1.0):.4f}")

# İkili: negatifler pozitiflerin üstünde → pozitif kayıp.
print(f"Marj (karışık):  {pairwise_margin_loss(neg, pos, margin=1.0):.4f}")


## 4. Uçtan uca: yeniden sıralama kazancı

Hattın tamamını birleştiriyoruz: HierarchicalRAG (Bi-Encoder erişimi) → CrossEncoderReranker (kesinlik için yeniden sıralama).

In [ ]:
from src.rag_engine.chunking import HierarchicalChunker
from src.rag_engine.pipeline import HierarchicalRAG
from src.rag_engine.vector_stores import build_store
from src.reranking.cross_encoder import CrossEncoderReranker
from src.reranking.pipeline import RerankingPipeline
import tempfile

persist = tempfile.mkdtemp()
rag = HierarchicalRAG(
    chunker=HierarchicalChunker(parent_chars=400, child_chars=180),
    encoder=SentenceEncoder("BAAI/bge-small-en-v1.5"),
    child_store=build_store("chroma", collection="rerank_demo", persist_dir=persist),
    parent_lookup={},
)

corpus = [
    ("hyde",     "HyDE generates a hypothetical answer document and embeds it for retrieval. The synthetic document lies closer to real answer documents than the original query does."),
    ("rerank",   "Cross-encoders score query-document pairs jointly via the transformer attention mechanism. They are slower than bi-encoders but substantially more accurate."),
    ("chunking", "Hierarchical chunking splits long documents into parent and child segments. Children are indexed for retrieval; parents are returned for context."),
    ("hnsw",     "HNSW is a graph-based approximate nearest neighbour algorithm. It maintains a hierarchy of small-world graphs and achieves logarithmic expected search time."),
    ("react",    "ReAct interleaves Thought, Action, and Observation steps. The language model verbalizes a plan, emits a tool call, and reads back the result."),
]
for doc_id, text in corpus:
    rag.ingest_document(text, metadata={"doc_id": doc_id, "title": doc_id})

reranker = RerankingPipeline(reranker=CrossEncoderReranker(), candidate_k=5)

query = "how do cross-encoders compare to bi-encoders?"
candidates = rag.retrieve(query, k=5)
print("BI-ENCODER İLK 5:")
for c in candidates:
    print(f"  [{c.score:+.3f}] {c.metadata['doc_id']}")

reranked = reranker.rerank(query, candidates, k=3)
print("\nCROSS-ENCODER İLK 3:")
for r in reranked:
    print(f"  [{r.cross_encoder_score:+.3f}] {r.metadata['doc_id']}")


**Az önce ne oldu.** Bi-Encoder, kosinüs benzerliklerine göre sıralanmış beş aday döndürdü. Cross-Encoder ise sorguya ve her belgeye birlikte dikkat ederek onları *yeniden puanladı* ve farklı (ve genellikle daha iyi) bir sıralama üretti. `docs/benchmarks.md` içindeki BEIR kıyaslama diliminde bu, nDCG@10'u 0,58'den 0,74'e çıkarır — ~85 ms ek gecikme karşılığında ~%28'lik göreli bir iyileşme.

## 5. Cohere'a geçiş

Aynı `Reranker` Protocol'ü barındırılan bir yeniden sıralayıcıyı da kabul eder. Cohere'a geçmek için:

```yaml
# configs/reranker_cohere.yaml
reranker:
  backend: cohere
  model: rerank-english-v3.0
  candidate_k: 50
```

```python
RerankingPipeline.from_config("configs/reranker_cohere.yaml")
```

Hat kodu değişmez. Bu, arayüze göre programlamanın getirisidir.

## 6. Alıştırmalar

1. **Zor negatif madenciliği.** Cross-encoder ince ayarı için *zor negatifler* bulmak üzere Bi-Encoder'ın ilk-k sonuçlarını kullanan bir betik yazın. Marj kaybı eğitim eğrilerinde yığın içi rastgele negatifleri zor negatiflerle karşılaştırın.
2. **HyDE ablasyonu.** 50 sorguluk küçük bir değerlendirme kümesi oluşturun. Erişim recall@10 değerini şunlarla karşılaştırın: (a) düz sorgu, (b) yeniden yazılmış sorgu, (c) HyDE ($K=1$), (d) HyDE ($K=4$). Hangi dönüşüm, maliyetini çıkardığı gecikmeye değer?
3. **Damıtma (distillation).** Küçük (3 katmanlı) bir cross-encoder'ı, logit'ler üzerinde MSE kullanarak büyük bir modelin skorlarını taklit edecek şekilde eğitin. Doğruluğun ne kadarı korunuyor? Ne kadar gecikme kazanılıyor?

## 7. Çıkarımlar

- Bi/Cross-Encoder ikiliği, modern erişimdeki en önemli tek mimari desendir. İki farklı hedef için optimize edilmiş iki aşama.
- Sorgu dönüşümü (yeniden yazma, ayrıştırma, HyDE) soru/yanıt manifoldu boşluğuna saldırır. HyDE, çıkarım anında bedavaya elde edilebilecek en büyük tek iyileştirmedir.
- Noktasal ve ikili kayıp arasındaki seçim, sistemin neye göre optimize edildiğini yansıtır: *sınıflandırma* mı, *sıralama* mı. Yeniden sıralama temelde bir sıralama görevidir.
- Yeniden sıralayıcıyı bir uygulama olarak değil, bir arayüz olarak ele alın. Sıfırdan bir CrossEncoder, bir Hugging Face modeli ve Cohere API'si aynı Protocol'ü karşılar.

➡ Gelecek hafta *erişimden* *ajanlığa* geçiyoruz: bir LLM'in bu erişim araçlarını bilinçli olarak kullanmasını sağlayan ReAct döngüsü.